In [ ]:
import shaker
import MDAnalysis as md
import nglview as nv

# Defining beads as Virtual sites with `shaker`

**[Virtual-sites](https://manual.gromacs.org/documentation/current/reference-manual/functions/interaction-methods.html#virtual-interaction-sites)** are massless particles whose positions
are not integrated directly by the equations of motion, but instead reconstructed
at every step from the positions of a set of real constructing atoms. Because they
carry no mass, they do not contribute to the kinetic energy or the integration
timestep — but they can carry charge, be involved in non-bonded interactions, or
represent interaction centers that do not coincide with any real atom.

**Virtual sites** are an important feature in coarse-grained models because they allow interaction centers to be positioned at physically meaningful locations **without introducing additional bonded interactions** required to constrain them. This is **particularly valuable for rigid molecules**, where reproducing the correct geometry using only bonds, angles, and dihedrals would often require many stiff potentials that can lead to numerical instability. By defining bead positions geometrically from neighboring particles, virtual sites enable accurate molecular shapes while maintaining stable and computationally efficient simulations.

In Martini, the most commonly used virtual sites are `type N` and `type 3` virtual sites.
- `Type N` virtual sites are defined from a set of reference particles and can be positioned at their center of geometry, center of mass, or a weighted average of their coordinates, making them useful for placing interaction sites at symmetric locations such as the center of aromatic rings. 
- `Type 3` virtual sites are defined by a weighted combination of three reference particles, allowing interaction centers to be positioned at arbitrary locations relative to the molecular framework.

Together, these virtual site types provide a flexible way to reproduce molecular geometry and interaction patterns without introducing additional bonded constraints, making them particularly valuable for rigid and highly structured molecules.

## Workflow with shaker

Shaker provides two functions to generate virtual site topology entries directly
from a CG structure:

- `generate_virtual_sites3` — for a rigid three-atom frame
- `generate_virtual_sitesN` — for a linear combination of N frame atoms

Both functions take an MDAnalysis `Universe` containing a **single averaged
reference structure** as input. The recommended way to obtain this is with
`align_mol_to_single_traj`, which aligns all molecules across all trajectory
frames into a single-molecule trajectory and computes the mean coordinates.

## 1. Building Type 3 Virtual sites with `shaker`

For our first example, we will use `shaker` to construct a [type 3 virtual-site](https://manual.gromacs.org/documentation/current/reference-manual/functions/interaction-methods.html#virtual-interaction-sites) representation that captures a molecules average structure. This greatly facilitates the parameterization of rigid molecules, such as cholesterol. The following showcase tutorial will follow the steps previously used to [parameterize the rigid steroid core of Cholesterol](https://pubs.acs.org/doi/10.1021/acs.jctc.3c00547).

As mentioned previously, `Type 3` virtual sites are defined by a weighted combination of three reference particles, allowing interaction centers to be positioned at arbitrary locations relative to the molecular framework.

The workflow consists of the following steps:
* **Produce a mapping**
* **Align all molecules in the trajectory & compute and average structure**
* **Define a 3 bead frame**
* **Construct the remaining beads as virtual sites**

### 1.1. Produce a mapping
This step follows the same `shaker` workflow described in the introductory parameterization tutorials. Cholesterol mapping follows the work described [elsewhere](https://pubs.acs.org/doi/10.1021/acs.jctc.3c00547).

In [ ]:
author  = 'Luis Borges-Araujo'
itp_header = [
    f"; Parameterised by {author} @ ENS de Lyon, 2026.\n"
    "; Molecular name: \n",
    "; SMILES: \n",
]


## AA reference sims (already PBC treated and removed solvent.)
GRO = '../AA_references/ComplexMembrane/pbc.gro'
XTC = '../AA_references/ComplexMembrane/shortpbc.xtc'

mapping = {
    ## Resname
    "CHL1": {
    # Bead name;  Mapping.
        "ROH": {"type": "P1",    "charge": 0,  "atoms": ['C2', 'C2', 'C4', 'C4', 'O3', 'O3', 'O3', 'O3'] },      
        "R1":  {"type": "SC4",   "charge": 0,  "atoms": ['C6', 'C6', 'C6', 'C6','C6', 'C5', 'C7','H6','H7A','H7B'] },             
        "R2":  {"type": "SC3",   "charge": 0,  "atoms": ['C9', 'H9', 'C10', 'C1','H1B','H1A','C9', 'H9', 'C10', 'C1','H1B','H1A','H1B','H1A','H1A','H1B'] }, 
        "R3":  {"type": "SC3",   "charge": 0,  "atoms": ['C15', 'C15', 'C15', 'C15', 'C15', 'C14','C14','C16','C16'] },      
        "R4":  {"type": "SC3",   "charge": 0,  "atoms": ['C11', 'C12'] }, 
        "R5":  {"type": "TC2",   "charge": 0,  "atoms": ['C19'] },              
        "R6":  {"type": "TC2",   "charge": 0,  "atoms": ['C18'] },              
        "C1":  {"type": "C2",    "charge": 0,  "atoms": ['C20','H22A','H22B','H20','H21A','H21B','H21C'] },
        "C2":  {"type": "C2",    "charge": 0,  "atoms": ['C23', 'C24', 'C25', 'C26', 'C27'] },
    },
}

shaker.mapper.map_aa2cg(GRO, XTC, mapping)

### 1.2. Align all molecules in the trajectory & compute an average structure
Since virtual site parameters are derived from a single reference geometry, we
first need to produce a representative average structure. For a rigid molecule
like cholesterol, this is straightforward: we align all copies of the molecule
across all trajectory frames onto a common reference, then average the resulting
coordinates.

`shaker` provides `align_mol_to_single_traj` for this purpose. It takes a
trajectory containing N molecules over X frames, aligns each molecule to a
chosen reference using a subset of atoms, and writes a single-molecule
trajectory of N×X frames. The average structure is then computed from this
aligned trajectory.

> **Note:** The `align_selection` should consist of atoms that are part of the
> rigid core of the molecule — in the case of cholesterol, beads from the
> steroid ring system are a natural choice. Flexible tails or terminal groups
> should be excluded, as their conformational variability would degrade the
> quality of the average structure.

In [ ]:
shaker.vsites.align_mol_to_single_traj('cg_mapped.gro', 'cg_mapped.xtc',
                    selection="resname CHOL CHL1 SITO ERG CAMP STIG",
                    align_selection="name C1 R2 R1",
                    reference_residue=0,)

In [ ]:
view = nv.show_mdanalysis(md.Universe('average_molecule.gro', 'aligned_molecules.xtc'))
view.clear_representations()
view.add_representation("spacefill", selection="all", radius=2.5)
view

### 1.3. Define a 3 bead frame

The three frame beads define the rigid coordinate system from which all other
bead positions will be reconstructed as virtual sites. The choice of frame is
important: 
- the three beads must be non-collinear (i.e. they must not lie on a
straight line);  
- they should span the molecule as broadly as possible to
minimize numerical sensitivity in the virtual site parameters.
- the distances between the frame beads, as well as the positions of all
  virtual sites relative to the frame, should show low variance across the
  trajectory — since the frame distances are enforced as rigid constraints and
  the virtual site parameters are fixed, high variance in any of these quantities
  indicates that the rigid body assumption does not hold for the chosen bead set.
- the three frame beads should form a triangle that is as **equilateral as 
  possible**; obtuse triangles — particularly configurations where two obtuse 
  triangles share an edge — result in highly coupled constraints that can cause 
  the LINCS algorithm to fail to converge, regardless of `lincs_iter` and 
  `lincs_order` settings.

For planar or near-planar molecules like the cholesterol steroid core, a good
frame consists of three beads that are well-separated and triangulate the rigid
scaffold. In practice, beads at opposite ends of the core and one off-axis bead
work well. 

**In this case we will use `['C1','R2',"R1"]`.**

The frame beads are the only particles that carry mass — all remaining beads
will be assigned zero mass as virtual sites. `shaker` handles this automatically
via the `mass_split="equal"` option, which redistributes the total molecular
mass evenly across the three frame beads.

> **Note:** Avoid choosing frame beads that are nearly collinear or very close
> together. A poorly conditioned frame will produce large virtual site
> coefficients that are sensitive to small structural fluctuations, and
> `shaker` will raise an error if the frame determinant falls below a safe
> threshold.

In [ ]:
frame_beads = ['C1','R2',"R1"] 

### 1.4. Construct the remaining beads as virtual sites.

Once the frame is defined, the remaining beads in the selection are expressed as
virtual sites whose positions are fully determined by the three frame beads. Since
virtual sites are massless by definition in GROMACS, their mass must be
redistributed onto the frame beads. `shaker` handles this automatically via the
`mass_split="equal"` option, which collects the total mass of all beads in the
selection and redistributes it evenly across the three frame beads, setting the
mass of all virtual sites to zero.

`generate_virtual_sites3` writes three topology blocks in a single call:

- **`[ constraints ]`** — fixes the distances between the three frame beads,
  enforcing the rigid geometry assumed by the virtual site construction.
- **`[ virtual_sites3 ]`** — defines the position of each remaining bead as a
  function of the frame, using the coefficients derived from the average structure.
- **`[ exclusions ]`** — excludes all non-bonded interactions between beads in
  the virtual site group. This is required because the constrained frame and the
  fixed virtual site geometry already fully determine the relative positions of
  these beads — allowing non-bonded interactions between them would double-count
  interactions that are implicitly encoded in the bonded topology.

All three sections must be present in the topology for the virtual site
construction to be physically correct. `shaker` generates them together to
ensure nothing is accidentally omitted.

The `updated_mapping` returned by the function contains the redistributed bead
masses, which can be passed downstream to the rest of the `shaker` parameterization
workflow.

> **Note:** The equal mass split is the only scheme currently supported by
> `shaker`. If your mapping requires a non-uniform mass distribution across the
> frame beads, the `updated_mapping` can be edited manually after the fact.

In [ ]:
u=md.Universe('average_molecule.gro')
lines, mapping = shaker.vsites.generate_virtual_sites3(u, frame_beads,
                                                       selection='not name C2', 
                                                       mass_split="equal",
                                                       mapping=mapping, resname='CHL1')

In [ ]:
lines

In [ ]:
shaker.itp.write_initial_CGitp("CHL1", mapping,
                               header = itp_header,
                               footer=lines)

In [ ]:
from pathlib import Path
print(Path('initial_CG.itp').read_text(), end='')

## 2. Building Type N Virtual sites with `shaker`

For our second example, we will use `shaker` to construct a
[type N virtual-site](https://manual.gromacs.org/documentation/current/reference-manual/functions/interaction-methods.html#virtual-interaction-sites)
representation using a linear combination of N frame beads. Compared to the
type 3 construction, this approach is conceptually simpler: rather than
expressing bead positions in a local coordinate frame via a basis decomposition,
each virtual site is placed at a weighted average of the constructing beads'
positions, with the weights derived directly from the reference geometry.

Among the several type N constructions supported by GROMACS (center of geometry,
center of mass, center of weights), `shaker` focuses exclusively on the
**center of weights** case. The center of geometry and center of mass cases
require no parameter derivation — the weights are either trivially equal or
directly given by the bead masses — and are straightforward to write by hand.
The center of weights case, by contrast, requires solving for the weights that
exactly reproduce each virtual site position from the reference structure, which
is what `shaker` automates.

The workflow mirrors that of the type 3 construction:

* **Produce a mapping**
* **Align all molecules in the trajectory & compute an average structure**
* **Define an N bead frame**
* **Construct the remaining beads as virtual sites**

To avoid repetition we will skip ahead the first two steps and start from an
already averaged CG structure of the tryptophan inositol sidechain. The
averaging and alignment procedure is identical to that described in
[Section 1.2](#12-align-all-molecules-in-the-trajectory--compute-an-average-structure).

### 2.1. Load and visualize CG structure

In [ ]:
cg ='../AA_references/VirtualSites/cg.pdb'

In [17]:
u=md.Universe(cg)
view = nv.show_mdanalysis(u)
view.clear_representations()
view.add_representation("spacefill", selection="all", radius=2.5)
view

NGLWidget()

### 2.2. Define an N bead frame

As with the type 3 construction, the frame beads define the rigid scaffold from
which all virtual site positions are reconstructed. However, for type N virtual
sites the bonded interactions that enforce the frame geometry — constraints,
bonds, and improper dihedrals — are defined manually by the user and are not
generated by `shaker`. The appropriate bonded terms are molecule-specific and
should reflect the known rigidity of the core.

For the tryptophan inositol sidechain, the frame follows the parameterization
described in the [Martini small molecules paper](https://advanced.onlinelibrary.wiley.com/doi/full/10.1002/adts.202100391),
where the rigid core is enforced by a combination of constraints and a single
bond and improper dihedral. This parameterization is not covered here.

Similar geometric considerations as for the type 3 frame
apply:

- the frame beads must be non-collinear;
- they should span the rigid core as broadly as possible;
- all inter-bead distances and relative positions should show low variance
  across the trajectory, validating the rigid body assumption;

**In this case, the frame consists of `['SC1','SC2',"SC4","SC5"]`.**

In [ ]:
frame_beads = ['SC1','SC2',"SC4","SC5"]

### 2.3. Construct the remaining beads as virtual sites

With the frame defined and its bonded interactions written manually, the
remaining beads can now be constructed as virtual sites using
`generate_virtual_sitesN`. The function returns two topology blocks —
`[ virtual_sitesn ]` and `[ exclusions ]` — which should be appended to the
manually written frame topology.

Note that `mass_split="equal"` is also available for type N virtual sites,
following the same logic as described in [Section 1.4](#14-construct-the-remaining-beads-as-virtual-sites).
It is not shown here for brevity.

It is worth noting that for a ring system like tryptophan, a center
of geometry construction — where all weights are equal — might seem like a
natural choice. However, the center of weights construction derived by `shaker`
from the averaged reference structure produces a more accurate placement of the
central bead, as the unequal weights reflect the true geometric centroid of the
rigid core rather than a simple average of bead positions. This difference is
small but meaningful for the accurate reproduction of interaction site positions.

In [ ]:
lines, _ = shaker.vsites.generate_virtual_sitesN(u, frame_beads)

In [ ]:
lines